# 🩺 Cuaderno 01: Análisis Exploratorio de Datos Acústicos (EDA)
> **Proyecto:** Sistema Embebido para Captura y Análisis de Señales Cardiorrespiratorias (*f tecno* - 8vo Semestre 2026)  
> **Objetivo:** Explorar las características espectrales y temporales de los sonidos pulmonares del dataset **ICBHI 2017**, comparando ciclos respiratorios normales contra patologías obstructivas e infecciosas (sibilancias y crepitantes).

--- 
## 1. Importación de Librerías y Carga del Dataset

In [ ]:
import sys
import os
from pathlib import Path

# Añadir directorio raíz al path
ROOT_DIR = Path.cwd().resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src.data_loader import load_icbhi_dataset, print_icbhi_summary
from src.audio_processing import load_audio_normalized, butter_bandpass_filter, segment_cycle

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Librerías importadas correctamente.")

--- 
## 2. Ingesta y Auditoría de Datos ICBHI Organizados

In [ ]:
df_icbhi = load_icbhi_dataset()
print_icbhi_summary(df_icbhi)
df_icbhi.head()

--- 
## 3. Distribución Estadística y Desbalance Clínico

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Distribución de clases por ciclo
class_counts = df_icbhi["cycle_class_name"].value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values, palette="Blues_d", ax=axes[0])
axes[0].set_title("Distribución de Ciclos Respiratorios (4 Clases)", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Número de Ciclos")
axes[0].set_xticklabels(class_counts.index, rotation=20, ha="right")

# 2. Diagnóstico a nivel de paciente
pat_df = df_icbhi.drop_duplicates(subset=["patient_id"])
diag_counts = pat_df["diagnosis"].value_counts()
sns.barplot(x=diag_counts.index, y=diag_counts.values, palette="Reds_d", ax=axes[1])
axes[1].set_title("Diagnóstico Clínico General (Nivel Paciente)", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Número de Pacientes")
axes[1].set_xticklabels(diag_counts.index, rotation=25, ha="right")

plt.tight_layout()
plt.show()

--- 
## 4. Visualización Comparativa Acústica: Normal vs Crepitantes vs Sibilancias

In [ ]:
# Seleccionar un ciclo representativo de cada clase
sample_normal = df_icbhi[df_icbhi["cycle_class"] == 0].iloc[0]
sample_crackles = df_icbhi[df_icbhi["cycle_class"] == 1].iloc[0]
sample_wheezes = df_icbhi[df_icbhi["cycle_class"] == 2].iloc[0]

samples = [
    ("Normal (Sin ruidos adventicios)", sample_normal),
    ("Crepitantes (Crackles - Neumonía/EPOC)", sample_crackles),
    ("Sibilancias (Wheezes - Obstrucción/Asma)", sample_wheezes)
]

fig, axes = plt.subplots(3, 2, figsize=(16, 10))
TARGET_SR = 16000

for idx, (title, row) in enumerate(samples):
    # Cargar y filtrar audio
    audio, sr = load_audio_normalized(row["audio_path"], target_sr=TARGET_SR)
    audio_filt = butter_bandpass_filter(audio, lowcut=100.0, highcut=2000.0, fs=TARGET_SR)
    cycle_audio = segment_cycle(audio_filt, row["start_time"], row["end_time"], sr=TARGET_SR, target_duration=4.0)
    
    # Forma de onda en el tiempo
    time_axis = np.linspace(0, 4.0, len(cycle_audio))
    axes[idx, 0].plot(time_axis, cycle_audio, color="royalblue", alpha=0.8)
    axes[idx, 0].set_title(f"{title} - Forma de Onda", fontweight="bold")
    axes[idx, 0].set_ylabel("Amplitud Norm.")
    axes[idx, 0].set_ylim([-1.05, 1.05])
    if idx == 2:
        axes[idx, 0].set_xlabel("Tiempo (segundos)")
        
    # Espectrograma Mel (Frecuencia vs Tiempo)
    mel_spec = librosa.feature.melspectrogram(y=cycle_audio, sr=TARGET_SR, n_fft=1024, hop_length=512, n_mels=64)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = librosa.display.specshow(mel_spec_db, sr=TARGET_SR, hop_length=512, x_axis="time", y_axis="mel", ax=axes[idx, 1], cmap="magma")
    axes[idx, 1].set_title(f"{title} - Espectrograma Mel", fontweight="bold")
    if idx == 2:
        axes[idx, 1].set_xlabel("Tiempo (segundos)")
        
plt.tight_layout()
plt.show()